In [ ]:
import warnings
from collections import defaultdict

import tensordict
import torch
import torch.nn.functional as F
from tensordict import TensorDictBase, TensorDict
from tensordict.nn import TensorDictModule, CompositeDistribution, set_interaction_type, InteractionType
from torch import distributions as d
from torch import multiprocessing, nn
from torch.nn.functional import one_hot
from torchrl.collectors import Collector
from torchrl.data import Composite, MultiOneHot, ReplayBuffer, LazyTensorStorage, \
    SamplerWithoutReplacement, UnboundedDiscrete, Binary
from torchrl.envs import (Compose, DoubleToFloat, StepCounter,
                          TransformedEnv, EnvBase, check_env_specs, RewardScaling)
from torchrl.modules import ValueOperator, ProbabilisticActor
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
import random

warnings.filterwarnings("ignore")

In [ ]:
is_fork = multiprocessing.get_start_method() == "fork"
device = torch.device("cpu")
lr = 3e-4  # Increased for better gradient flow in PPO
max_grad_norm = 0.5
sub_batch_size = 32
num_epochs = 4
clip_epsilon = 0.15
gamma = 0.995
lmbda = 0.95
entropy_eps = 0.01  # Reduced from 0.1 - was causing instability
bc_lr = 5e-4
bc_max_grad_norm = 0.5
head_weight_clip = 0.25
tensordict.nn.set_composite_lp_aggregate(False).set()

def sanitize_logits(logits: torch.Tensor, min_logit: float = -8.0) -> torch.Tensor:
    logits = torch.nan_to_num(logits, nan=min_logit, posinf=0.0, neginf=min_logit)
    logits = logits - logits.detach().amax(dim=-1, keepdim=True)
    return torch.clamp(logits, min=min_logit, max=0.0)

def is_finite_tensor(x: torch.Tensor) -> bool:
    return torch.isfinite(x).all().item()

In [ ]:
frames_per_batch = 1000
total_frames = 50_000

In [ ]:
from torch import argmax


class SkyJoEnv(EnvBase):
    metadata = {"render.modes": ["human"]}
    
    def __init__(self, device: torch.device):
        super().__init__(device=device)
        self._make_spec()
        self.setup()

    def _make_spec(self):
        self.observation_spec = Composite(
            # Twelve table cards, each from -2 to 12 + unknown + gone and -2 to 12 for the top of the discard pile
            state=Binary(shape=12 * 17 + 15, dtype=torch.float32)
        )
        self.action_spec = MultiOneHot((2, 12))
        self.reward_spec = UnboundedDiscrete()

    def _step(self, tensordict: TensorDictBase) -> TensorDictBase:
        action = tensordict["action"]
        if isinstance(action, TensorDictBase):
            action_pile = argmax(one_hot(action["pile"], 2))
            action_card = argmax(one_hot(action["card"], 12))
        else:
            action_pile = argmax(action[:2])
            action_card = argmax(action[2:])
        state, reward, done = self.game_step(action_pile, action_card)
        state = torch.tensor(snap.state(), dtype=torch.float32)
        reward = torch.nan_to_num(torch.tensor(snap.reward(), dtype=torch.float32), nan=0.0, posinf=100.0, neginf=-100.0)
        reward = torch.clamp(reward, -100.0, 100.0)
        out = TensorDict({
            "state": state,
            "reward": reward.unsqueeze(-1),
            "done": torch.tensor(snap.done(), dtype=torch.bool).unsqueeze(-1),
        }, tensordict.shape)
        return out
    
    def game_step(self, pile: int, card: int) -> tuple[torch.Tensor, int, bool]:
        
        pass

    def _reset(self, tensordict: TensorDictBase, **kwargs) -> TensorDictBase:
        self.env.reset()
        out = TensorDict({
            "state": torch.tensor(self.env.state(), dtype=torch.float32),
        }, batch_size=self.batch_size)
        return out
    
    def setup(self):
        cards = [-2] * 3 + [0] * 11 + [-1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] * 7
        random.shuffle(cards)
        self.table_cards = [cards.pop() for _ in range(12)]
        self.discard_top = cards.pop()
        self.draw_pile = cards
        

    def _set_seed(self, seed: int | None) -> None:
        # ignore for now
        pass


In [ ]:
base_env = SkyJoEnv(device=device)
env = TransformedEnv(
    base_env,
    Compose([
        DoubleToFloat(),
        StepCounter(),
    ])
)

check_env_specs(env)

In [ ]:
class BaseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.spec = env.output_spec
        self.net1 = nn.Sequential(
            nn.Linear(12 * 17 + 15, 128),
            nn.LayerNorm(128),
            nn.ELU(),
            nn.Dropout(0.1),
        )
        self.net2 = nn.Sequential(
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.ELU(),
            nn.Dropout(0.1),
        )
        self.net3 = nn.Sequential(
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ELU(),
            nn.Dropout(0.1),
        )
            

    def forward(self, x: torch.Tensor):
        x = self.net1(x)
        res = x
        x = self.net2(x)
        x = x + res
        x = self.net3(x)
        return x


In [ ]:
class ActorNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = BaseNetwork()
        self.head_0 = nn.Linear(64, 2)  # pile
        torch.nn.init.xavier_uniform_(self.head_0.weight, gain=0.001)
        torch.nn.init.zeros_(self.head_0.bias)
        self.head_1 = nn.Linear(64, 12)  # table card
        torch.nn.init.xavier_uniform_(self.head_1.weight, gain=0.001)
        torch.nn.init.zeros_(self.head_1.bias)

    def forward(self, state):
        state = sanitize_state(state)
        features = torch.nan_to_num(self.base(state), nan=0.0, posinf=1e3, neginf=-1e3)
        # Compute logits with careful numerical stability
        l0 = self.head_0(features)
        l1 = self.head_1(features)

        l0 = sanitize_logits(l0)
        l1 = sanitize_logits(l1)

        return l0, l1


class CriticNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = BaseNetwork()
        self.head = nn.Linear(64, 1)

    def forward(self, state):
        state = sanitize_state(state)
        features = torch.nan_to_num(self.base(state), nan=0.0, posinf=1e3, neginf=-1e3)
        value = self.head(features)
        return torch.clamp(torch.nan_to_num(value, nan=0.0, posinf=100.0, neginf=-100.0), -100.0, 100.0)

In [ ]:
actor_module = TensorDictModule(
    ActorNet(),
    in_keys=["state"],
    out_keys=[
        ("params", "pile", "logits"),
        ("params", "card", "logits"),
    ],
)

actor = ProbabilisticActor(
    module=actor_module,
    in_keys=["params"],
    out_keys=[
        ("action", "pile"),
        ("action", "card"),
    ],
    distribution_class=CompositeDistribution,
    distribution_kwargs={
        "distribution_map": {
            "pile": d.Categorical,
            "card": d.Categorical,
        },
        "name_map": {
            "pile": ("action", "pile"),
            "card": ("action", "card"),
        },
    },
    return_log_prob=True,
)

# Add a wrapper to catch inf/nan in probabilities
original_actor_forward = actor.forward
def actor_forward_safe(td):
    if "state" in td.keys():
        td["state"] = sanitize_state(td["state"])
    try:
        result = original_actor_forward(td)
        # Check logits for inf/nan
        for key in [("params", "movement", "logits"), ("params", "rotation", "logits"), ("params", "additional", "logits")]:
            if key in td:
                logits = td[key]
                if torch.isnan(logits).any() or torch.isinf(logits).any():
                    print(f"WARNING: inf/nan in logits {key}: min={logits.min()}, max={logits.max()}")
                    # Replace with safe values
                    td[key] = sanitize_logits(logits)
        return result
    except RuntimeError as e:
        if "categorical" in str(e).lower() or "probability" in str(e).lower():
            print(f"CRITICAL: {e}")
            print(f"Logits stats:")
            for key in [("params", "movement", "logits"), ("params", "rotation", "logits"), ("params", "additional", "logits")]:
                if key in td:
                    logits = td[key]
                    print(f"  {key}: min={logits.min()}, max={logits.max()}, mean={logits.mean()}, any_nan={torch.isnan(logits).any()}, any_inf={torch.isinf(logits).any()}")
                    td[key] = sanitize_logits(logits)
            raise
        raise

actor.forward = actor_forward_safe

critic = ValueOperator(
    CriticNet(),
    in_keys=["state"],
    out_keys=["state_value"],
)

In [ ]:
collector = Collector(
    env,
    actor,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames,
    device=device,
)

replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=frames_per_batch),
    sampler=SamplerWithoutReplacement(),
)

advantage_module = GAE(
    gamma=gamma, lmbda=lmbda, value_network=critic, average_gae=True, device=device,
)

loss_module = ClipPPOLoss(
    actor_network=actor,
    critic_network=critic,
    clip_epsilon=clip_epsilon,
    entropy_bonus=bool(entropy_eps),
    entropy_coeff=entropy_eps,
    normalize_advantage=True,  # Normalize advantages for better numerical stability
)

optim = torch.optim.Adam(loss_module.parameters(), lr, weight_decay=1e-5)

In [ ]:
logs = defaultdict(list)
eval_str = ""
it = 0

# We iterate over the collector until it reaches the total number of frames it was
# designed to collect:
for i, tensordict_data in enumerate(collector):
    if "state" in tensordict_data.keys():
        tensordict_data["state"] = sanitize_state(tensordict_data["state"])
    if ("next", "state") in tensordict_data.keys(True):
        tensordict_data["next", "state"] = sanitize_state(tensordict_data["next", "state"])
    if ("next", "reward") in tensordict_data.keys(True):
        rew = torch.nan_to_num(tensordict_data["next", "reward"], nan=0.0, posinf=100.0, neginf=-100.0)
        tensordict_data["next", "reward"] = torch.clamp(rew, -100.0, 100.0)
    # we now have a batch of data to work with. Let's learn something from it.
    with torch.no_grad():
        advantage_module(tensordict_data)

         # Normalize advantages very aggressively to prevent explosion
        advantage = tensordict_data.get("advantage", None)
        if advantage is not None:
            # First clip extreme outliers
            advantage = torch.nan_to_num(advantage, nan=0.0, posinf=100.0, neginf=-100.0)
            advantage = torch.clamp(advantage, -100.0, 100.0)
            # Then normalize
            adv_mean = advantage.mean()
            adv_std = advantage.std() + 1e-8
            tensordict_data["advantage"] = (advantage - adv_mean) / adv_std
            # Final safety clip to reasonable range
            tensordict_data["advantage"] = torch.clamp(tensordict_data["advantage"], -5.0, 5.0)

    data_view = tensordict_data.reshape(-1)
    print("Max reward: ", tensordict_data["next", "reward"].max().item())

    # Additional safety check on logits that were generated during collection
    invalid_batch = False
    if ("params", "movement", "logits") in tensordict_data:
        for key in [("params", "movement", "logits"), ("params", "rotation", "logits"), ("params", "additional", "logits")]:
            logits = tensordict_data[key]
            if torch.isnan(logits).any() or torch.isinf(logits).any():
                print(f"WARNING: inf/nan detected in collected {key}")
                print(f"  Stats: min={logits.min()}, max={logits.max()}, mean={logits.mean()}")
                # Don't use this batch to avoid poisoning the replay buffer
                invalid_batch = True
                break
            tensordict_data[key] = sanitize_logits(logits)

    if "sample_log_prob" in tensordict_data.keys(True):
        slp = tensordict_data["sample_log_prob"]
        tensordict_data["sample_log_prob"] = torch.clamp(torch.nan_to_num(slp, nan=-8.0, posinf=0.0, neginf=-8.0), -8.0, 0.0)

    if invalid_batch:
        continue

    replay_buffer.extend(data_view.cpu())
    for _ in range(num_epochs):
        for f in range(frames_per_batch // sub_batch_size):
            subdata = replay_buffer.sample(sub_batch_size)
            if "state" in subdata.keys():
                subdata["state"] = sanitize_state(subdata["state"])
            if ("next", "state") in subdata.keys(True):
                subdata["next", "state"] = sanitize_state(subdata["next", "state"])
            if ("next", "reward") in subdata.keys(True):
                rew = torch.nan_to_num(subdata["next", "reward"], nan=0.0, posinf=100.0, neginf=-100.0)
                subdata["next", "reward"] = torch.clamp(rew, -100.0, 100.0)

            # Clip advantages to prevent extreme values
            advantage = subdata.get("advantage", None)
            if advantage is not None:
                # Aggressively clip advantages to prevent explosion
                advantage = torch.nan_to_num(advantage, nan=0.0, posinf=10.0, neginf=-10.0)
                subdata["advantage"] = torch.clamp(advantage, -10.0, 10.0)

            if "sample_log_prob" in subdata.keys(True):
                slp = subdata["sample_log_prob"]
                subdata["sample_log_prob"] = torch.clamp(torch.nan_to_num(slp, nan=-8.0, posinf=0.0, neginf=-8.0), -8.0, 0.0)

            loss_vals = loss_module(subdata.to(device))
            if (not torch.isfinite(loss_vals["loss_objective"])) or (not torch.isfinite(loss_vals["loss_critic"])) or (not torch.isfinite(loss_vals["loss_entropy"])):
                print(f"WARNING: non-finite PPO components: objective={loss_vals['loss_objective']}, critic={loss_vals['loss_critic']}, entropy={loss_vals['loss_entropy']}")
                continue
            loss_value = (
                    loss_vals["loss_objective"]
                    + loss_vals["loss_critic"]
                    + loss_vals["loss_entropy"]
            )

            # Check for NaN/Inf before backprop
            if torch.isnan(loss_value) or torch.isinf(loss_value):
                print(f"WARNING: NaN/Inf detected in loss! loss_objective={loss_vals['loss_objective']}, "
                      f"loss_critic={loss_vals['loss_critic']}, loss_entropy={loss_vals['loss_entropy']}")
                # Print debug info
                if advantage is not None:
                    print(f"  Advantage stats: min={advantage.min()}, max={advantage.max()}, mean={advantage.mean()}")
                continue

            optim.zero_grad()
            loss_value.backward()

            bad_grad = False
            for p in loss_module.parameters():
                if p.grad is not None and (not torch.isfinite(p.grad).all()):
                    bad_grad = True
                    break
            if bad_grad:
                print("WARNING: non-finite PPO gradients, skipping update")
                optim.zero_grad()
                continue

            grad_norm = torch.nn.utils.clip_grad_norm_(loss_module.parameters(), max_grad_norm)
            if not torch.isfinite(grad_norm):
                print("WARNING: non-finite PPO grad norm, skipping update")
                optim.zero_grad()
                continue
            optim.step()

    logs["reward"].append(tensordict_data["next", "reward"].mean().item())
    it += tensordict_data.numel()
    cum_reward_str = (
        f"it {it}, average reward={logs['reward'][-1]: 4.4f}"
    )
    logs["step_count"].append(tensordict_data["step_count"].max().item())
    stepcount_str = f"step count (max): {logs['step_count'][-1]}"
    if i % 5 == 0:
        with set_interaction_type(InteractionType.DETERMINISTIC), torch.no_grad():
            eval_rollout = env.rollout(1000, actor)
            td = eval_rollout[0]  # single step
            actor_module.eval()
            actor_module(td)  # run just the TensorDictModule part
            actor_module.train()

            print("movement probs: ", F.softmax(td[("params", "movement", "logits")], dim=-1))
            print("rotation probs: ", F.softmax(td[("params", "rotation", "logits")], dim=-1))
            print("additional probs:", F.softmax(td[("params", "additional", "logits")], dim=-1))

            logs["eval reward"].append(eval_rollout["next", "reward"].mean().item())
            logs["eval reward (sum)"].append(
                eval_rollout["next", "reward"].mean().item()
            )
            logs["eval step_count"].append(eval_rollout["step_count"].max().item())
            eval_str = (
                f"eval  reward: {logs['eval reward (sum)'][-1]: 4.4f} "
                f"eval step-count: {logs['eval step_count'][-1]}"
            )
            del eval_rollout

    print(", ".join([eval_str, cum_reward_str, stepcount_str]))
